In [1]:
# 08/2025
###########################################################################################
# using linear model to test group difference
#	Testing group difference
#   Plotting results
#   Diagnostics of the model
#   non-parametric alternative if necessary (not yet implemented)

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

In [3]:
curProject = 'amputee'
curRegion = 'CSSyl'  # CSSyl or CSpreCS
curRoot = 'C'  # 'C' or 'D'
curDistType = 'min'         ############################ !!!!!!!!!!!!!!!  CHANGE  !!!!!!!!!!!!!!  ###########################

################################  Read the CSV file into a DataFrame  ################################
#file_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curRegion}_combined_{curDistType}_2024.csv'
#file_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curRegion}_withMorphometry_{curDistType}_2024.csv'
file_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curRegion}_withMorphometry_asy_{curDistType}_2024.csv'
#file_path = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\{curRegion}_combined_{curDistType}_2025.csv'
print(file_path)

merged_info = pd.read_csv(file_path)
print(len(merged_info))
#print("Data types:\n", df_loaded.dtypes)

# Add numeric columns for tests that require numeric columns, eg ANCOVA
merged_info['Gender_num'] = merged_info['Gender'].map({'M': 0, 'F': 1})    # Create the new column 'Category_num'
merged_info['Hemisphere_num'] = merged_info['Hemisphere'].map({'Left': 1, 'Right': 2})    # Create the new column 'Category_num'
#merged_info['Study_num'] = merged_info['Study'].map({'1': 1, '2': 2})    # Create the new column 'Category_num'
merged_info['Study_num'] = pd.to_numeric(merged_info['Study'], errors='coerce')  # Study is already stored as integer


# Make sure categorical variables are treated as such
merged_info['Group'] = merged_info['Group'].astype('category')
merged_info['Gender'] = merged_info['Gender'].astype('category')
merged_info['Study'] = merged_info['Study'].astype('category')


C:\B_projWIP\proj_amputee\Analysis_2025\CSSyl_withMorphometry_asy_min_2024.csv
130


In [4]:
#print(merged_info['Study'])
print(merged_info['Study'].unique())
print(merged_info['Study'].dtype)
print(merged_info['Study_num'].dtype)

[2, 1]
Categories (2, int64): [1, 2]
category
int64


In [5]:
#merged_info.isna().sum()
nan_counts = merged_info.isna().sum().sort_values(ascending=False)
print(nan_counts)

Prosthesisusage    82
Myo                82
Cos                82
amputatio level    82
Stumpusage         82
                   ..
Hemisphere          0
Gender              0
AgeScan             0
Group               0
Study_num           0
Length: 66, dtype: int64


In [6]:
################################  Data Selection  ################################
left_hem = merged_info[merged_info['Hemisphere'] == 'Left']
right_hem = merged_info[merged_info['Hemisphere'] == 'Right']
controls = merged_info[merged_info['Group'] == 'CTR'] 

# DataFrame with missing hemisphere:
missing_hem = merged_info[
    ((merged_info['Hemisphere'] == 'Left') & (merged_info['AmpSide'] == 'R')) |
    ((merged_info['Hemisphere'] == 'Right') & (merged_info['AmpSide'] == 'L'))
]

# DataFrame with using hemisphere:
using_hem = merged_info[
    ((merged_info['Hemisphere'] == 'Left') & (merged_info['AmpSide'] == 'L')) |
    ((merged_info['Hemisphere'] == 'Right') & (merged_info['AmpSide'] == 'R'))
]

missing_hem_cong = missing_hem[missing_hem['Group'] == 'CONG']
using_hem_cong = using_hem[using_hem['Group'] == 'CONG']

missing_hem_withControl = pd.concat([missing_hem, controls], ignore_index=True)
using_hem_withControl = pd.concat([using_hem, controls], ignore_index=True)

missing_hem_cong_control = pd.concat([missing_hem_cong, controls], ignore_index=True)
using_hem_cong_control = pd.concat([using_hem_cong, controls], ignore_index=True)


In [7]:
"""
## remove unused groups
## NOTE: not removing unused categories could cause problems doing stats

missing_hem_cong_control['Group'] = missing_hem_cong_control['Group'].cat.remove_unused_categories()
using_hem_cong_control['Group'] = using_hem_cong_control['Group'].cat.remove_unused_categories()
"""
#############  checking selections  #############
missing_hem_cong_control['Group'].unique()
using_hem_cong_control['Group'].unique()

['CONG', 'CTR']
Categories (3, object): ['AMP', 'CONG', 'CTR']

In [8]:
######################################  Parametric linear model  #######################################

In [9]:
############################ !!!!!!!!!!!!!!!  CHANGE  !!!!!!!!!!!!!!  ###########################
# Define the dataset
# using_hem_withControl, missing_hem_withControl, using_hem, missing_hem, right_hem, left_hem, merged_info
# using_hem_cong_control, missing_hem_cong_control
cur_info = missing_hem_withControl  

In [10]:
####################  testing using two categories instead of three  ####################

cur_info['Group_redefined'] = cur_info['Group'].replace({
    'AMP': 'AMP_CTR',
    'CTR': 'AMP_CTR',
    'CONG': 'CONG'
})

# If Group is categorical, clean up unused categories
cur_info['Group_redefined'] = cur_info['Group_redefined'].astype('category')
cur_info['Group_redefined'] = cur_info['Group_redefined'].cat.remove_unused_categories()

print(cur_info['Group_redefined'].unique())

['CONG', 'AMP_CTR']
Categories (2, object): ['AMP_CTR', 'CONG']


C:\Users\joyca\AppData\Local\Temp\ipykernel_69188\2934639630.py:3: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  cur_info['Group_redefined'] = cur_info['Group'].replace({


In [11]:
#######################  LM tests, parametric  ########################
# List of shape measures
#shape_measures = ['iso1', 'iso2', 'iso3', 'UMAP1_U1', 'UMAP1_U2', 'UMAP1_U3', 'UMAP1_U4','UMAP2_U3','UMAP2_U4']
#shape_measures = ['surface_talairach','maxdepth_talairach','meandepth_talairach','hull_junction_length_talairach',
#                     'GM_thickness','opening','iso1','UMAP1_U2',
#                  'surface_talairach_asy','maxdepth_talairach_asy','meandepth_talairach_asy','hull_junction_length_talairach_asy',
#                     'GM_thickness_asy','opening_asy','iso1_asy','UMAP1_U2_asy'
#                    ]   
shape_measures = ['surface_talairach_LRasy','maxdepth_talairach_LRasy','meandepth_talairach_LRasy','hull_junction_length_talairach_LRasy',
                     'GM_thickness_LRasy','opening_LRasy','iso1_LRasy','UMAP1_U2_LRasy'
                    ]  

# Dictionary to store results
model_results = {}

for measure in shape_measures:
    formula = f'{measure} ~ Group + AgeScan + Gender + Study'
    #formula = f'{measure} ~ Group + Hemisphere'
    #formula = f'{measure} ~ Group + Study + Hemisphere'
    #formula = f'{measure} ~ Group_redefined + AgeScan + Gender + Study'    # testing using two categories
    model = smf.ols(formula, data=cur_info).fit()
    model_results[measure] = model
    print(f'========================= Results for {measure} ===========================')
    print(model.summary())
    print('\n')

========================= Results for surface_talairach_LRasy ===========================
                               OLS Regression Results                              
Dep. Variable:     surface_talairach_LRasy   R-squared:                       0.108
Model:                                 OLS   Adj. R-squared:                  0.055
Method:                      Least Squares   F-statistic:                     2.015
Date:                     Wed, 17 Dec 2025   Prob (F-statistic):             0.0849
Time:                             12:01:28   Log-Likelihood:                 136.78
No. Observations:                       89   AIC:                            -261.6
Df Residuals:                           83   BIC:                            -246.6
Df Model:                                5                                         
Covariance Type:                 nonrobust                                         
                    coef    std err          t      P>|t|      [0.025 

In [12]:
#########################################  Quick confirmation  ##########################################
shape_measures = ['iso1'] # 'iso1', 'UMAP1_U2'

# Dictionary to store results
model_results = {}

for measure in shape_measures:
    formula = f'{measure} ~ Group + Hemisphere + AgeScan + Gender + Study'
    #formula = f'{measure} ~ Group + Hemisphere + Study'  
    #formula = f'{measure} ~ Group + Hemisphere'
    
    model = smf.ols(formula, data=cur_info).fit()
    model_results[measure] = model
    print(f'========================= Results for {measure} ===========================')
    print(model.summary())
    print('\n')

========================= Results for iso1 ===========================
                            OLS Regression Results                            
Dep. Variable:                   iso1   R-squared:                       0.179
Model:                            OLS   Adj. R-squared:                  0.119
Method:                 Least Squares   F-statistic:                     2.974
Date:                Wed, 17 Dec 2025   Prob (F-statistic):             0.0112
Time:                        12:01:28   Log-Likelihood:                -219.62
No. Observations:                  89   AIC:                             453.2
Df Residuals:                      82   BIC:                             470.7
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------

In [13]:
##############################  naive post hoc with no covariates  ####################################
# Tukey HSD
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Tukey HSD on residualized values
tukey = pairwise_tukeyhsd(
    endog=cur_info['UMAP1_U2'],    # dependent variable
    groups=cur_info['Group'],      # group labels
    alpha=0.05                     # significance level
)

print(tukey)


Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
   AMP   CONG    3.094 0.0005   1.216   4.972   True
   AMP    CTR  -0.0187 0.9996  -1.712  1.6746  False
  CONG    CTR  -3.1127    0.0 -4.5594 -1.6659   True
----------------------------------------------------


In [14]:
##############################  post hoc with covariates  ####################################
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def run_models_with_posthoc(data, measures, covariates=['AgeScan', 'Gender', 'Study','Hemisphere']):
    results = []

    for measure in measures:
        # Build formula
        formula = f"{measure} ~ Group + {' + '.join(covariates)}"
        model = smf.ols(formula, data=data).fit()
        # Get the "Corrected" data (Residuals)
        # Add the Group effect back to the residuals to compare groups 
        # while 'leveling the playing field' such as Age/Gender
        corrected_values = model.resid + model.params['Intercept'] 

        # Run Tukey HSD on Group
        tukey = pairwise_tukeyhsd(
            #endog=data[measure],    # if data is already residualized
            endog=corrected_values, # if raw data, following the correction above           
            groups=data['Group'],
            alpha=0.05
        )        
        # Store results in DataFrame
        tukey_df = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])
        tukey_df["Measure"] = measure  # tag which dependent variable

        results.append(tukey_df)

    # Combine all results
    return pd.concat(results, ignore_index=True)

#measures = ["iso1", "UMAP1_U2", "UMAP2_U4"]  # min, missing_hem
measures = ["opening"]  # GM_thickness,opening, min, missing_hem
posthoc_results = run_models_with_posthoc(cur_info, measures)

print("\nAll Post-hoc Results:")
print(posthoc_results)


All Post-hoc Results:
  group1 group2  meandiff  p-adj   lower   upper  reject  Measure
0    AMP   CONG      -0.0    1.0 -0.3752  0.3752   False  opening
1    AMP    CTR      -0.0    1.0 -0.3383  0.3383   False  opening
2   CONG    CTR       0.0    1.0 -0.2891  0.2891   False  opening


In [15]:
#import sys
#!{sys.executable} -m pip install --upgrade pingouin

#!conda install -c conda-forge pingouin -y

#!pip install --upgrade pingouin

import pingouin as pg
print(pg.__version__)

0.5.5


In [16]:
# Check for zero variance in covariates
for cov in ['AgeScan', 'Gender', 'Study']:
    unique_counts = cur_info[cov].nunique()
    print(f"Column '{cov}' has {unique_counts} unique value(s).")
    if unique_counts < 2:
        print(f"⚠️ WARNING: '{cov}' is a constant! This will cause Division by Zero.")

Column 'AgeScan' has 34 unique value(s).
Column 'Gender' has 2 unique value(s).
Column 'Study' has 2 unique value(s).


In [26]:
######################  work around of pingouin version freeze problem  #######################
# versions older ( <= 0.5.13 ) pg.pairwise_tests doesn't accept covar as a parameter

import pingouin as pg

def run_proper_ancova_robust(data, measure, group_col, covariates):
    # 1. Clean data: Drop any rows where measure or group is missing
    df_temp = data.dropna(subset=[measure, group_col]).copy()
    
    # Ensure group_col is treated as a string
    df_temp[group_col] = df_temp[group_col].astype(str)
    
    # --- Step 1: Filter out "Dead" Covariates (Constant values) ---
    valid_covs = []
    for cov in covariates:
        if df_temp[cov].nunique() > 1:
            valid_covs.append(cov)
        else:
            print(f"Skipping covariate '{cov}' because it has only 1 unique value.")
    
    # --- Step 2: Dummies for Categorical Covariates ---
    cat_covs = [c for c in valid_covs if df_temp[c].dtype == 'object']
    if cat_covs:
        df_temp = pd.get_dummies(df_temp, columns=cat_covs, drop_first=True)
    
    # Rebuild the final covariate list for the formula
    final_covariates = [c for c in valid_covs if c not in cat_covs]
    for cat in cat_covs:
        final_covariates.extend([col for col in df_temp.columns if col.startswith(cat + '_')])

    # --- Step 3: Clean "Ghost" Groups ---
    counts = df_temp[group_col].value_counts()
    valid_groups = counts[counts >= 2].index.tolist()
    df_temp = df_temp[df_temp[group_col].isin(valid_groups)].copy()

    if len(valid_groups) < 2:
        print("Not enough valid groups (N >= 2) to perform a comparison.")
        return None, None

    # --- Step 4: Run the math ---
    print(f"--- Running ANCOVA with covariates: {final_covariates} ---")
    
    # Main ANCOVA
    ancova_res = pg.ancova(data=df_temp, dv=measure, covar=final_covariates, between=group_col)
    
    # Adjusted Post-hoc via Residuals
    # This formula regresses out the covariates
    formula = f"{measure} ~ {' + '.join(final_covariates)}"
    model = smf.ols(formula, data=df_temp).fit()
    
    # Create the adjusted value column
    df_temp['adj_val'] = model.resid + df_temp[measure].mean()

    # CRITICAL FIX: Use the string "none" instead of None
    posthoc = pg.pairwise_tests(
        data=df_temp, 
        dv='adj_val', 
        between=group_col, 
        padjust='bonf',
        effsize="none" 
    )
    
    return ancova_res, posthoc

# Run
# Note: I'm using your specific covariate names from the previous prompt
ancova, posthoc = run_proper_ancova_robust(cur_info, 'opening', 'Group', ['AgeScan', 'Gender_num', 'Study_num'])
print(ancova)
print(posthoc)

--- Running ANCOVA with covariates: ['AgeScan', 'Gender_num', 'Study_num'] ---
       Source         SS  DF          F     p-unc       np2
0       Group   1.609194   2   3.148035  0.048106  0.070508
1     AgeScan   4.928230   1  19.281997  0.000033  0.188518
2  Gender_num   0.586873   1   2.296177  0.133492  0.026920
3   Study_num   0.114083   1   0.446358  0.505923  0.005349
4    Residual  21.213731  83        NaN       NaN       NaN
  Contrast     A     B  Paired  Parametric         T        dof alternative  \
0    Group   AMP  CONG   False        True  0.552028  26.974625   two-sided   
1    Group   AMP   CTR   False        True  1.736590  18.183337   two-sided   
2    Group  CONG   CTR   False        True  1.632131  36.397822   two-sided   

      p-unc    p-corr p-adjust   BF10      none  
0  0.585474  1.000000     bonf  0.352  0.185710  
1  0.099371  0.298113     bonf  0.971  0.660925  
2  0.111275  0.333825     bonf  0.776  0.450374  


In [18]:
######################  Use EMMeans for post hoc  #######################
#import pingouin as pg

def run_proper_ancova(data, measure, group_col, covariates):
    """
    Performs ANCOVA and post-hoc tests adjusted for covariates.
    """
    
    # Run ANCOVA to check if 'Group' is significant while controlling for covariates
    ancova_res = pg.ancova(data=data, dv=measure, covar=covariates, between=group_col)
    print(f"--- ANCOVA Results for {measure} ---")
    print(ancova_res)

    # Run Post-Hoc with Bonferroni or Tukey adjustment
    # Pingouin's pairwise_tests handles the covariate adjustment automatically
    # It calculates "Estimated Marginal Means" (EMMeans)
    posthoc = pg.pairwise_tests(
        data=data, 
        dv=measure, 
        between=group_col, 
        covar=covariates, 
        padjust='bonf'  # choices: 'bonf', 'sidak', or 'fdr_bh'
    )
    
    print(f"\n--- Adjusted Post-Hoc Results ---")
    return posthoc

curMeasure = 'opening'
covariates = ['AgeScan', 'Gender_num', 'Study_num','Hemisphere_num']
results = run_proper_ancova(cur_info, curMeasure, 'Group', covariates)

--- ANCOVA Results for opening ---
           Source         SS  DF          F     p-unc       np2
0           Group   1.916622   2   3.783357  0.026814  0.084481
1         AgeScan   5.029106   1  19.854622  0.000026  0.194931
2      Gender_num   0.670242   1   2.646075  0.107643  0.031260
3       Study_num   0.091160   1   0.359896  0.550218  0.004370
4  Hemisphere_num   0.443418   1   1.750590  0.189480  0.020902
5        Residual  20.770312  82        NaN       NaN       NaN


TypeError: pairwise_tests() got an unexpected keyword argument 'covar'

In [ ]:
######################################  Adjusted Means (Marginal Means) Plot  ######################################
measure = "iso2"
formula = f'{measure} ~ Group + AgeScan + Gender + Study'
model = smf.ols(formula, data=cur_info).fit()

# Get adjusted means (marginal effects)
marg_means = sm.stats.anova_lm(model, typ=2)
print(marg_means)

# Estimated marginal means (predicted values by group)
means = model.get_prediction(
    cur_info.assign(AgeScan=cur_info.AgeScan.mean(),  # hold covariates at mean
                    Gender=cur_info.Gender.mode()[0],
                    Study=cur_info.Study.mode()[0])
).summary_frame()

# Add back Group info
df_pred = cur_info[['Group']].copy()
df_pred['pred'] = means['mean']
df_pred['ci_low'] = means['mean_ci_lower']
df_pred['ci_high'] = means['mean_ci_upper']

# Plot group means with error bars
plt.figure(figsize=(6,4))
sns.pointplot(data=df_pred, x='Group', y='pred', 
              join=False, capsize=.1, errwidth=1.5, color="black")
plt.ylabel("Adjusted iso2")
plt.title("Group differences in iso2 (adjusted for covariates)")
sns.despine()
plt.show()


In [ ]:
#####################  Boxplot of Raw Data  #####################
measure = 'opening'      # opening, GM_thickness,  # <-- choose the measure you want
group_col = 'Group'

plt.figure(figsize=(5, 4))
sns.boxplot(
    data=cur_info,
    x=group_col,
    y=measure,
    palette='Set2'
)
sns.stripplot(
    data=cur_info,
    x=group_col,
    y=measure,
    color='black',
    alpha=0.5,
    jitter=True
)
plt.title(f'{measure} by Group')
plt.xlabel('Group')
plt.ylabel(measure)

curPlotName = rf'{curRoot}:\B_projWIP\proj_{curProject}\Analysis_2025\Plots\2024_{measure}_{curRegion}_{curDistType}_congAmpCtr_Box_Both.png'
print(curPlotName)
#plt.savefig(curPlotName)

plt.show()


In [ ]:
#####################  Violin of Raw Data  #####################

plt.figure(figsize=(6,4))
sns.violinplot(data=cur_info, x="Group", y="iso2", inner="box", palette="Set2")
plt.title("Raw distribution of iso2 across groups")
plt.ylabel("iso2")
sns.despine()
plt.show()


In [ ]:
model = None

############################  Model diagnostics  #############################
#	Linearity: residuals vs. fitted plot should have no curve

# Fit model
formula = "UMAP1_U2 ~ Group + AgeScan + Gender + Study"
model = smf.ols(formula, data=cur_info).fit()

# Get fitted values and residuals
fitted_vals = model.fittedvalues
residuals = model.resid

# Plot residuals vs. fitted
plt.scatter(fitted_vals, residuals, alpha=0.7)
plt.axhline(y=0, color='r', linestyle='--', linewidth=1)
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals vs. Fitted")
plt.show()


In [ ]:
#	Homoscedasticity: scale-location plot (residuals vs. fitted with equal spread)
# Get fitted values and standardized residuals
fitted_vals = model.fittedvalues
residuals = model.resid
standardized_resid = residuals / np.std(residuals)

# Square root of absolute standardized residuals
sqrt_abs_resid = np.sqrt(np.abs(standardized_resid))

plt.scatter(fitted_vals, sqrt_abs_resid, alpha=0.7)
sm.nonparametric.lowess(sqrt_abs_resid, fitted_vals)  # add smoothing line
plt.plot(fitted_vals, sm.nonparametric.lowess(sqrt_abs_resid, fitted_vals)[:, 1], color="red")

plt.xlabel("Fitted values")
plt.ylabel("√|Standardized residuals|")
plt.title("Scale-Location Plot")
plt.show()

In [ ]:
############################  For linearity: Ramsey RESET test  ##############################
# Look at the Residuals vs. fitted (no systematic curvature).
# Ramsey RESET test (general test of mis-specification):
# If p < 0.05 → possible nonlinearity / omitted variables.
    
from statsmodels.stats.diagnostic import linear_reset

reset_test = linear_reset(model, power=2, use_f=True)
print(reset_test)


In [ ]:
################### Testing heteroscedasticity  ####################
# Breusch–Pagan test
# if p < 0.05, heteroscedasticity might exist
from statsmodels.stats.diagnostic import het_breuschpagan

bp_test = het_breuschpagan(model.resid, model.model.exog)
labels = ['Lagrange multiplier statistic', 'p-value', 
          'f-value', 'f p-value']
print(dict(zip(labels, bp_test)))


In [ ]:
################### Testing heteroscedasticity  ####################
# White test (more general, allows for nonlinearities)
from statsmodels.stats.diagnostic import het_white

white_test = het_white(model.resid, model.model.exog)
labels = ['LM Statistic', 'LM-Test p-value', 'F-Statistic', 'F-Test p-value']
print(dict(zip(labels, white_test)))


In [ ]:
#	Normality of residuals: Q–Q plot, 

# Get residuals from your fitted model
residuals = model.resid

# Q–Q plot
sm.qqplot(residuals, line='45', fit=True)
plt.title("Q–Q Plot of Residuals")
plt.show()

In [ ]:
# Shapiro–Wilk test (though with large n, visual inspection is better)
# Why “visual inspection is better with large n”?
# With a large sample size (say, n > 50–100), the Shapiro–Wilk test becomes too sensitive: 
#    it will flag tiny, irrelevant deviations as significant.
#    in practice, researchers rely more on Q–Q plots and also consider robustness of their model 
#       (linear regression is fairly robust to mild non-normality if sample size is big enough)

from scipy.stats import shapiro
stat, p = shapiro(residuals)
print('Shapiro-Wilk Test statistic=%.3f, p-value=%.3f' % (stat, p))

if p > 0.05:
    print("Residuals look normally distributed (fail to reject H0).")
else:
    print("Residuals are not normally distributed (reject H0).")

In [ ]:
###################  Heteroscedasticity Alternative Model  ####################
# Use robust standard errors:
robust_model = model.get_robustcov_results(cov_type='HC3')
print(robust_model.summary())

In [ ]:
print(cur_info.columns)

In [ ]:
########################################  Influence studies  ########################################
## Cook's distance

# Influence measures
influence = model.get_influence()
cooks_d, pvals = influence.cooks_distance

# Subject IDs
#subject_ids = cur_info['SubjID'].astype(str).tolist()   # display SubjID
#subject_ids = cur_info.index.astype(str).tolist()       # display index, to compare with the influence plot below
subject_ids = cur_info['subjName'].astype(str).tolist()  # default: display subjName

# Plot Cook's distance
plt.figure(figsize=(10, 6))
plt.stem(range(len(cooks_d)), cooks_d, markerfmt=",")  # no use_line_collection

# Custom red line: default = 4/n
n = len(cooks_d)
threshold = 4/n
plt.axhline(y=threshold, color='r', linestyle='--', label=f'Threshold={threshold:.3f}')

# Annotate IDs for influential points
for i, d in enumerate(cooks_d):
    if d > threshold:
        plt.text(i, d, subject_ids[i], fontsize=8, ha='right', va='bottom')

plt.xlabel("Observation index")
plt.ylabel("Cook's distance")
plt.title("Cook's Distance Plot with Subject IDs")
plt.legend()
plt.show()


In [ ]:
# Create a mapping DataFrame
index_map = cur_info.reset_index()[['index', 'subjName', 'SubjID']]

# Rename the columns for clarity
index_map.columns = ['RowIndex', 'SubjName', 'SubjID']

pd.set_option('display.max_rows', None)
print(index_map)


In [ ]:
########################################  Influence studies  ########################################
## Leverage vs. residuals squared plot
sm.graphics.influence_plot(model, criterion="cooks")
plt.show()

In [ ]:
############################################  WIP  ###############################################

In [ ]:
#########################  Non-parametric alternative (if needed) ##########################
